In [1]:
pip install numpy pandas scipy statsmodels openpyxl

In [19]:
import numpy as np
import pandas as pd
import scipy.io as sio
import statsmodels.api as sm
from datetime import datetime
import os

# --- 1. 基础工具函数 ---

def load_mat_data(file_name, variable_name):
    try:
        mat = sio.loadmat(file_name)
        return mat[variable_name]
    except FileNotFoundError:
        print(f"错误：文件 {file_name} 未找到。")
        return None

def matlab_datenum_to_datetime(matlab_datenum):
    timestamps = (matlab_datenum - 719529) * 86400 * 1000
    return pd.to_datetime(timestamps, unit='ms')

def ols_hac_matlab_style(y, x, lags=5):
    """
    复刻 MATLAB olsgmm 的逻辑: b=(X'X)\(X'y), Newey-West SE.
    """
    X_df = pd.DataFrame(x).copy()
    X_df.columns = ['X_Factor']
    X_with_const = sm.add_constant(X_df, prepend=True)

    data = pd.concat([y, X_with_const], axis=1).dropna()

    if data.empty or len(data) <= X_with_const.shape[1]:
        return np.nan, np.nan, np.nan, 0

    Y_clean = data.iloc[:, 0]
    X_clean = data.iloc[:, 1:]

    model = sm.OLS(Y_clean, X_clean)
    results = model.fit(cov_type='HAC', cov_kwds={'maxlags': lags, 'kernel': 'bartlett'})

    if 'X_Factor' not in results.params:
        return np.nan, np.nan, np.nan, results.nobs

    beta = results.params['X_Factor']
    se_beta = results.bse['X_Factor']
    r2_adj = results.rsquared_adj

    return beta, se_beta, r2_adj, results.nobs

# --- 2. 主逻辑 ---

def run_strict_replication_final_v9():
    # 参数设置
    date_begin_str = '1975-01-31'
    date_end_str = '2015-12-31'

    ScalingFactor = 12
    ScalingFactor_Real = 1200
    NW_lags = 5

    countries = ['Australia', 'Canada', 'Germany', 'Japan', 'New Zealand',
                 'Norway', 'Sweden', 'Switzerland', 'United Kingdom', 'United States']
    us_idx = 9

    # 硬编码 IMF Map
    imf_map = {
        'Australia': 193, 'Canada': 156, 'Germany': 134, 'Japan': 158,
        'New Zealand': 196, 'Norway': 142, 'Sweden': 144, 'Switzerland': 146,
        'United Kingdom': 112, 'United States': 111
    }

    print("Loading Data...")
    gfd_loc = load_mat_data('Bonds_local_M.mat', 'GFD_Bonds_local_M')
    gfd_dol = load_mat_data('Bonds_dollar_M.mat', 'GFD_Bonds_dollar_M')
    gfd_tb  = load_mat_data('TB_M.mat', 'GFD_TB_M')
    gfd_yld = load_mat_data('Yields_M.mat', 'GFD_Yields_M')

    if any(x is None for x in [gfd_loc, gfd_dol, gfd_tb, gfd_yld]):
        return

    # --- 1. 智能切片与列提取 ---

    def apply_slice(arr, slice_obj):
        return np.concatenate([arr[:, s] for s in slice_obj], axis=1)

    # Yields: [1:45, 47:54, 56:end] (MATLAB 1-based) -> Python: 0:45, 46:54, 55:end
    raw_yld = apply_slice(gfd_yld, [slice(0, 45), slice(46, 54), slice(55, gfd_yld.shape[1])])

    # TBILL: [1:13, 15:52, 54:56, 58:end] -> Python: 0:13, 14:52, 53:56, 57:end
    raw_tb = apply_slice(gfd_tb, [slice(0, 13), slice(14, 52), slice(53, 56), slice(57, gfd_tb.shape[1])])

    # Bonds (Local & Dollar): [1:7, 9:end] -> Python: 0:7, 8:end
    raw_loc = apply_slice(gfd_loc, [slice(0, 7), slice(8, gfd_loc.shape[1])])
    raw_dol = apply_slice(gfd_dol, [slice(0, 7), slice(8, gfd_dol.shape[1])])

    # --- 2. 鲁棒的国家提取 ---

    def get_g10_data(cleaned_mat, debug_name=""):
        codes = cleaned_mat[0, :]
        dates = matlab_datenum_to_datetime(cleaned_mat[:, 0])

        data_cols = []
        for c in countries:
            target = imf_map[c]
            # 使用 isclose 解决浮点数匹配问题 (如 158.0 vs 158.00001)
            match_mask = np.isclose(codes, target, atol=1e-5)
            match_idx = np.where(match_mask)[0]

            if len(match_idx) > 0:
                data_cols.append(cleaned_mat[:, match_idx[0]])
            else:
                if debug_name:
                    print(f"Warning [{debug_name}]: Code {target} for {c} NOT FOUND in sliced matrix.")
                data_cols.append(np.full(cleaned_mat.shape[0], np.nan))

        df = pd.DataFrame(np.column_stack(data_cols), index=dates, columns=countries)

        # 排序并去重索引 (防止 KeyError)
        df.sort_index(inplace=True)
        if df.index.duplicated().any():
            df = df[~df.index.duplicated(keep='first')]

        return df

    df_loc = get_g10_data(raw_loc, "Local Bonds")
    df_dol = get_g10_data(raw_dol, "Dollar Bonds")
    df_tb  = get_g10_data(raw_tb, "TBills")
    df_yld = get_g10_data(raw_yld, "Yields")

    # 时间对齐
    valid_dates = df_loc.index
    start_mask = valid_dates >= pd.to_datetime(date_begin_str)
    if not start_mask.any(): return
    start_loc = np.argmax(start_mask)
    line_begin = max(0, start_loc - 1)

    TB = df_tb.iloc[line_begin:]
    Yield = df_yld.iloc[line_begin:]
    B_loc = df_loc.iloc[line_begin:]
    B_dol = df_dol.iloc[line_begin:]

    # --- 3. 变量计算 ---

    TB_month = (TB / TB.shift(1)) - 1
    BondsHPR = (B_loc / B_loc.shift(1)) - 1
    BondsHPRDollar = (B_dol / B_dol.shift(1)) - 1

    Yield_t_minus_1 = Yield.shift(1)
    Yield_Monthly_Log = np.log(1 + Yield_t_minus_1 / 100) / 12
    TB_month_Log = np.log(1 + TB_month)
    Slope = Yield_Monthly_Log - TB_month_Log

    Rf_US = TB_month.iloc[:, us_idx]

    log_R_dol = np.log(1 + BondsHPRDollar)
    log_R_loc = np.log(1 + BondsHPR)
    log_Rf_US = np.log(1 + Rf_US)
    log_Rf_Loc = np.log(1 + TB_month)

    BondsHPERDollar = log_R_dol.sub(log_Rf_US, axis=0)
    BondsHPER = log_R_loc - log_Rf_Loc
    FXEXR = BondsHPERDollar - BondsHPER

    BondsHPRDiff = BondsHPER.sub(BondsHPER.iloc[:, us_idx], axis=0)
    TotalDollarDiff = BondsHPERDollar.sub(BondsHPERDollar.iloc[:, us_idx], axis=0)

    RateDiff = TB_month.sub(TB_month.iloc[:, us_idx], axis=0)
    SlopeDiff = Slope.sub(Slope.iloc[:, us_idx], axis=0)

    # 过滤时间段 (N=492)
    mask = (TotalDollarDiff.index >= pd.to_datetime(date_begin_str)) & \
           (TotalDollarDiff.index <= pd.to_datetime(date_end_str))

    Y1 = TotalDollarDiff[mask]
    Y2 = FXEXR[mask]
    Y3 = BondsHPRDiff[mask]
    XA = RateDiff[mask]
    XB = SlopeDiff[mask]

    countries_reg = countries[:-1]

    # --- 4. 回归与输出 ---

    # >>> Panel A <<<
    print("\n" + "="*80)
    print("Table 1 Panel A: Interest Rate Differentials")
    print(f"Scaling: X * {ScalingFactor_Real}, Y * {ScalingFactor_Real}")
    print("="*80)
    print(f"{'Country':<15} | {'Total ($) Diff'} | {'FX Return'} | {'Local Bond Diff'}")
    print(f"{'':<15} | {'Beta':>6} {'SE':>6} {'R2':>6} | {'Beta':>6} {'SE':>6} {'R2':>6} | {'Beta':>6} {'SE':>6} {'R2':>6}")
    print("-" * 105)

    nobs_final = 0
    for i, c in enumerate(countries_reg):
        x = XA.iloc[:, i] * ScalingFactor_Real

        b1, se1, r1, n = ols_hac_matlab_style(Y1.iloc[:, i]*ScalingFactor_Real, x, NW_lags)
        if n > 0: nobs_final = n

        b2, se2, r2, _ = ols_hac_matlab_style(Y2.iloc[:, i]*ScalingFactor_Real, x, NW_lags)
        b3, se3, r3, _ = ols_hac_matlab_style(Y3.iloc[:, i]*ScalingFactor_Real, x, NW_lags)

        print(f"{c:<15} | {b1:6.2f} {se1:6.2f} {r1*100:6.1f} | {b2:6.2f} {se2:6.2f} {r2*100:6.1f} | {b3:6.2f} {se3:6.2f} {r3*100:6.1f}")

    print(f"N = {nobs_final}")

    # >>> Panel B <<<
    print("\n" + "="*80)
    print("Table 1 Panel B: Slope Differentials")
    print(f"Scaling: X * {ScalingFactor}, Y * {ScalingFactor}")
    print("="*80)
    print(f"{'Country':<15} | {'Total ($) Diff'} | {'FX Return'} | {'Local Bond Diff'}")
    # 🔴 新增：Panel B 的子标题行
    print(f"{'':<15} | {'Beta':>6} {'SE':>6} {'R2':>6} | {'Beta':>6} {'SE':>6} {'R2':>6} | {'Beta':>6} {'SE':>6} {'R2':>6}")
    print("-" * 105)

    for i, c in enumerate(countries_reg):
        x = XB.iloc[:, i] * ScalingFactor
        y_scale = ScalingFactor

        b1, se1, r1, _ = ols_hac_matlab_style(Y1.iloc[:, i]*y_scale, x, NW_lags)
        b2, se2, r2, _ = ols_hac_matlab_style(Y2.iloc[:, i]*y_scale, x, NW_lags)
        b3, se3, r3, _ = ols_hac_matlab_style(Y3.iloc[:, i]*y_scale, x, NW_lags)

        print(f"{c:<15} | {b1:6.2f} {se1:6.2f} {r1*100:6.1f} | {b2:6.2f} {se2:6.2f} {r2*100:6.1f} | {b3:6.2f} {se3:6.2f} {r3*100:6.1f}")

    print(f"N = {nobs_final}")

if __name__ == "__main__":
    run_strict_replication_final_v9()

<>:24: SyntaxWarning: invalid escape sequence '\('
<>:24: SyntaxWarning: invalid escape sequence '\('
/tmp/ipython-input-2925718931.py:24: SyntaxWarning: invalid escape sequence '\('
  复刻 MATLAB olsgmm 的逻辑: b=(X'X)\(X'y), Newey-West SE.


Loading Data...

Table 1 Panel A: Interest Rate Differentials
Scaling: X * 1200, Y * 1200
Country         | Total ($) Diff | FX Return | Local Bond Diff
                |   Beta     SE     R2 |   Beta     SE     R2 |   Beta     SE     R2
---------------------------------------------------------------------------------------------------------
Australia       |  -0.15   0.97   -0.2 |   1.28   0.61    0.6 |  -1.43   0.60    1.5
Canada          |  -1.10   0.69    0.1 |   1.21   0.52    0.5 |  -2.31   0.45    3.6
Germany         |   1.51   1.20    0.4 |   2.48   0.98    1.7 |  -0.97   0.60    0.5
Japan           |   2.35   0.84    1.1 |   3.09   0.67    3.5 |  -0.74   0.52    0.1
New Zealand     |   0.68   0.86   -0.0 |   2.21   0.48    3.1 |  -1.52   0.65    1.6
Norway          |   0.72   0.62    0.1 |   1.73   0.57    2.3 |  -1.02   0.41    1.0
Sweden          |  -0.64   0.91   -0.0 |   0.88   0.90    0.3 |  -1.52   0.49    2.0
Switzerland     |   1.15   0.82    0.3 |   2.44   0.77    2.4